# E-Commerce Business Performance Analysis

## Objective
Analyze 1M+ transactions from a UK-based e-commerce retailer (2009–2011) 
to identify revenue trends, product performance, and market concentration risks.

## Dataset
- Source: Online Retail II UCI Dataset (Kaggle)
- Size: 1,067,371 transactions
- Period: December 2009 – December 2011

## Analysis Steps
1. Data loading and exploration
2. Data quality check
3. Data cleaning
4. KPI analysis
5. Export for Power BI dashboard

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('../data/online_retail_II.csv')

print("Dataset size:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Dataset size: (1067371, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [22]:
# Data quality check
print(f"Total countries: {df['Country'].nunique()}")
print(f"Total products: {df['StockCode'].nunique()}")
print(f"Total customers: {df['Customer ID'].nunique()}")

Total countries: 43
Total products: 5305
Total customers: 5942


In [23]:
# Data cleaning
df_clean = df.copy()

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Add Revenue column
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# Separate returns from sales
df_returns = df_clean[df_clean['Quantity'] < 0].copy()
df_sales = df_clean[df_clean['Quantity'] > 0].copy()

# Remove rows with missing Customer ID for customer analysis
df_customers = df_sales.dropna(subset=['Customer ID']).copy()
df_customers['Customer ID'] = df_customers['Customer ID'].astype(int)

# Remove free items (Price = 0)
df_sales = df_sales[df_sales['Price'] > 0]

# Add time columns
df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month'] = df_sales['InvoiceDate'].dt.month
df_sales['YearMonth'] = df_sales['InvoiceDate'].dt.strftime('%Y-%m')

print("=== After Cleaning ===")
print(f"Total sales transactions: {len(df_sales):,}")
print(f"Total returns transactions: {len(df_returns):,}")
print(f"Total revenue: ${df_sales['Revenue'].sum():,.2f}")
print(f"Date range: {df_sales['InvoiceDate'].min().date()} to {df_sales['InvoiceDate'].max().date()}")
print(f"\nRevenue by Year:")
print(df_sales.groupby('Year')['Revenue'].sum().apply(lambda x: f"${x:,.2f}"))

=== After Cleaning ===
Total sales transactions: 1,041,671
Total returns transactions: 22,950
Total revenue: $20,972,968.14
Date range: 2009-12-01 to 2011-12-09

Revenue by Year:
Year
2009       $825,685.76
2010    $10,304,325.97
2011     $9,842,956.40
Name: Revenue, dtype: object


In [24]:
# Monthly revenue trend
monthly_revenue = df_sales.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)

# Top 10 products by revenue
top_products = df_sales.groupby('Description')['Revenue'].sum()\
    .sort_values(ascending=False).head(10).reset_index()
top_products.columns = ['Product', 'Revenue']

# Revenue by country (top 10)
top_countries = df_sales.groupby('Country')['Revenue'].sum()\
    .sort_values(ascending=False).head(10).reset_index()

# Monthly order volume
monthly_orders = df_sales.groupby('YearMonth')['Invoice'].nunique().reset_index()
monthly_orders['YearMonth'] = monthly_orders['YearMonth'].astype(str)
monthly_orders.columns = ['YearMonth', 'Orders']

# Average order value
aov = df_sales.groupby('Invoice')['Revenue'].sum().mean()

print("=== Key Business KPIs ===")
print(f"Total Revenue: ${df_sales['Revenue'].sum():,.2f}")
print(f"Total Orders: {df_sales['Invoice'].nunique():,}")
print(f"Average Order Value: ${aov:,.2f}")
print(f"Total Customers: {df_customers['Customer ID'].nunique():,}")
print(f"Return Rate: {len(df_returns)/(len(df_sales)+len(df_returns))*100:.1f}%")

print("\n=== Top 5 Products by Revenue ===")
print(top_products.head())

print("\n=== Top 5 Countries by Revenue ===")
print(top_countries.head())

=== Key Business KPIs ===
Total Revenue: $20,972,968.14
Total Orders: 40,078
Average Order Value: $523.30
Total Customers: 5,881
Return Rate: 2.2%

=== Top 5 Products by Revenue ===
                              Product    Revenue
0            REGENCY CAKESTAND 3 TIER  344563.25
1                              Manual  341104.90
2                      DOTCOM POSTAGE  322657.48
3  WHITE HANGING HEART T-LIGHT HOLDER  266923.55
4         PAPER CRAFT , LITTLE BIRDIE  168469.60

=== Top 5 Countries by Revenue ===
          Country       Revenue
0  United Kingdom  1.787135e+07
1            EIRE  6.644318e+05
2     Netherlands  5.542323e+05
3         Germany  4.312625e+05
4          France  3.569446e+05


In [25]:
# Remove non-product items
exclude_keywords = ['Manual', 'POSTAGE', 'DOTCOM', 'Discount', 
                    'AMAZONFEE', 'Bank Charges', 'CRUK']

mask = ~df_sales['Description'].str.upper().str.contains(
    '|'.join(exclude_keywords), na=False)

df_sales_clean = df_sales[mask].copy()

print(f"After removing non-products: {len(df_sales_clean):,} transactions")

# Recalculate top products
top_products_clean = df_sales_clean.groupby('Description')['Revenue'].sum()\
    .sort_values(ascending=False).head(10).reset_index()
top_products_clean.columns = ['Product', 'Revenue']

print("\n=== Top 5 Products (Cleaned) ===")
print(top_products_clean.head())

# Export for Power BI
df_sales_clean.to_csv('../exports/sales_clean.csv', index=False)
monthly_revenue.to_csv('../exports/monthly_revenue.csv', index=False)
top_products_clean.to_csv('../exports/top_products.csv', index=False)
top_countries.to_csv('../exports/top_countries.csv', index=False)

# KPI summary
kpi_summary = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Orders', 'Average Order Value', 
               'Total Customers', 'Return Rate', 'Total Countries'],
    'Value': [
        f"${df_sales_clean['Revenue'].sum():,.0f}",
        f"{df_sales_clean['Invoice'].nunique():,}",
        f"${df_sales_clean.groupby('Invoice')['Revenue'].sum().mean():,.2f}",
        f"{df_customers['Customer ID'].nunique():,}",
        f"{len(df_returns)/(len(df_sales)+len(df_returns))*100:.1f}%",
        f"{df_sales_clean['Country'].nunique()}"
    ]
})

kpi_summary.to_csv('../exports/kpi_summary.csv', index=False)

print("\n All files exported to exports/")
print("   - sales_clean.csv")
print("   - monthly_revenue.csv")
print("   - top_products.csv")
print("   - top_countries.csv")
print("   - kpi_summary.csv")

After removing non-products: 1,038,258 transactions

=== Top 5 Products (Cleaned) ===
                              Product    Revenue
0            REGENCY CAKESTAND 3 TIER  344563.25
1                              Manual  341104.90
2  WHITE HANGING HEART T-LIGHT HOLDER  266923.55
3         PAPER CRAFT , LITTLE BIRDIE  168469.60
4             JUMBO BAG RED RETROSPOT  150935.56

 All files exported to exports/
   - sales_clean.csv
   - monthly_revenue.csv
   - top_products.csv
   - top_countries.csv
   - kpi_summary.csv


In [26]:
# Check what "Manual" actually is
manual_check = df_sales[df_sales['Description'] == 'Manual'][['Invoice', 'Description', 'Price', 'Quantity', 'Revenue']].head(10)
print(manual_check)
print(f"\nTotal Manual revenue: ${df_sales[df_sales['Description'] == 'Manual']['Revenue'].sum():,.2f}")

      Invoice Description    Price  Quantity  Revenue
2697   489609      Manual     4.00         1     4.00
11310  490300      Manual     0.85         1     0.85
11311  490300      Manual     0.21         1     0.21
17386  490760      Manual    10.00         1    10.00
17887  490881      Manual    10.00         1    10.00
19542  490999      Manual    15.95         1    15.95
19543  490999      Manual     8.70         1     8.70
22960  491176      Manual  1213.02         1  1213.02
37762  492484      Manual     0.35         4     1.40
38527  492586      Manual     1.25         1     1.25

Total Manual revenue: $341,104.90


In [27]:
# Fix filtering - remove Manual and other non-product items
exclude_exact = ['Manual', 'DOTCOM POSTAGE', 'AMAZONFEE', 
                 'Bank Charges', 'CRUK', 'Discount']

exclude_contains = ['POSTAGE', 'AMAZONFEE', 'BANK CHARGES']

# Exact match filter
mask1 = ~df_sales['Description'].isin(exclude_exact)

# Contains filter
mask2 = ~df_sales['Description'].str.upper()\
    .str.contains('|'.join(exclude_contains), na=False)

df_sales_clean = df_sales[mask1 & mask2].copy()

# Recalculate top products
top_products_clean = df_sales_clean.groupby('Description')['Revenue'].sum()\
    .sort_values(ascending=False).head(10).reset_index()
top_products_clean.columns = ['Product', 'Revenue']

print(f"After cleaning: {len(df_sales_clean):,} transactions")
print("\n=== Top 5 Products (Cleaned) ===")
print(top_products_clean.head())

# Re-export
df_sales_clean.to_csv('../exports/sales_clean.csv', index=False)
top_products_clean.to_csv('../exports/top_products.csv', index=False)
print("\n Files re-exported")

After cleaning: 1,037,427 transactions

=== Top 5 Products (Cleaned) ===
                              Product    Revenue
0            REGENCY CAKESTAND 3 TIER  344563.25
1  WHITE HANGING HEART T-LIGHT HOLDER  266923.55
2         PAPER CRAFT , LITTLE BIRDIE  168469.60
3             JUMBO BAG RED RETROSPOT  150935.56
4                       PARTY BUNTING  149187.05

 Files re-exported


In [28]:
# Monthly revenue trend (fix YearMonth for export)
monthly_revenue = df_sales_clean.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)
monthly_revenue.columns = ['YearMonth', 'Revenue']

# Monthly orders
monthly_orders = df_sales_clean.groupby('YearMonth')['Invoice']\
    .nunique().reset_index()
monthly_orders['YearMonth'] = monthly_orders['YearMonth'].astype(str)
monthly_orders.columns = ['YearMonth', 'Orders']

# Merge
monthly_summary = monthly_revenue.merge(monthly_orders, on='YearMonth')
monthly_summary['AOV'] = monthly_summary['Revenue'] / monthly_summary['Orders']

print("=== Monthly Summary (Last 6 months) ===")
print(monthly_summary.tail(6).to_string(index=False))

# Re-export
monthly_summary.to_csv('../exports/monthly_revenue.csv', index=False)

# Final KPI summary
total_revenue = df_sales_clean['Revenue'].sum()
total_orders = df_sales_clean['Invoice'].nunique()
aov = df_sales_clean.groupby('Invoice')['Revenue'].sum().mean()
total_customers = df_customers['Customer ID'].nunique()
return_rate = len(df_returns)/(len(df_sales)+len(df_returns))*100

kpi_summary = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Orders', 
               'Average Order Value', 'Total Customers', 
               'Return Rate', 'Total Countries'],
    'Value': [
        round(total_revenue, 2),
        total_orders,
        round(aov, 2),
        total_customers,
        round(return_rate, 1),
        df_sales_clean['Country'].nunique()
    ]
})

kpi_summary.to_csv('../exports/kpi_summary.csv', index=False)
top_countries.to_csv('../exports/top_countries.csv', index=False)

print("\n All files ready for Power BI")
print("   - sales_clean.csv")
print("   - monthly_revenue.csv")
print("   - top_products.csv")
print("   - top_countries.csv")
print("   - kpi_summary.csv")

=== Monthly Summary (Last 6 months) ===
YearMonth     Revenue  Orders        AOV
  2011-07  689947.741    1452 475.170621
  2011-08  737067.220    1341 549.639985
  2011-09 1031400.361    1819 567.015042
  2011-10 1107419.540    2006 552.053609
  2011-11 1458895.530    2751 530.314624
  2011-12  615701.600     816 754.536275

 All files ready for Power BI
   - sales_clean.csv
   - monthly_revenue.csv
   - top_products.csv
   - top_countries.csv
   - kpi_summary.csv


In [29]:
# UK vs Others
uk_revenue = df_sales_clean[df_sales_clean['Country'] == 'United Kingdom']['Revenue'].sum()
others_revenue = df_sales_clean[df_sales_clean['Country'] != 'United Kingdom']['Revenue'].sum()

uk_vs_others = pd.DataFrame({
    'Market': ['United Kingdom', 'Others'],
    'Revenue': [uk_revenue, others_revenue]
})

print(uk_vs_others)
uk_vs_others.to_csv('../exports/uk_vs_others.csv', index=False)
print("Exported")

           Market       Revenue
0  United Kingdom  1.729136e+07
1          Others  2.889317e+06
Exported
